# Práctica — Clase 8: perros vs. gatos

Completa los dos ejercicios. Cada uno tiene celdas con `# TODO:` y una solución colapsada debajo.

In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds

TAMANO_IMAGEN = (128, 128)

(dataset_entreno, dataset_validacion), info = tfds.load(
    "cats_vs_dogs",
    split=["train[:80%]", "train[80%:]"],
    as_supervised=True,
    with_info=True,
)

def preprocesar(imagen, etiqueta):
    imagen = tf.image.resize(imagen, TAMANO_IMAGEN) / 255.0
    return imagen, etiqueta

dataset_entreno = dataset_entreno.map(preprocesar).batch(32).prefetch(1)
dataset_validacion = dataset_validacion.map(preprocesar).batch(32).prefetch(1)

## Ejercicio 1 — Completa el aumento de datos

**Enunciado**: define `aumento_datos` con tres capas: volteo horizontal aleatorio, rotación aleatoria de 0.1, y zoom aleatorio de 0.1.

**Pista**: las capas se llaman `RandomFlip`, `RandomRotation` y `RandomZoom`.

In [ ]:
aumento_datos = tf.keras.Sequential([
    # TODO: añade RandomFlip("horizontal")
    # TODO: añade RandomRotation(0.1)
    # TODO: añade RandomZoom(0.1)
])

modelo = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(128, 128, 3)),
    aumento_datos,
    tf.keras.layers.Conv2D(32, 3, activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])
modelo.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
modelo.fit(dataset_entreno, epochs=3, validation_data=dataset_validacion)

<details>
<summary>💡 Ver solución</summary>

```python
aumento_datos = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
])
```
</details>

## Ejercicio 2 — Cuenta los aciertos y errores del modelo

**Enunciado**: completa la función `contar_aciertos` para que compare, lote a lote del conjunto de validación, la predicción del modelo (umbral 0.5) con la etiqueta real, y devuelva el número total de aciertos y de fallos.

**Pista**: una predicción es correcta cuando `(predicción >= 0.5)` coincide con la etiqueta real (0 o 1).

In [ ]:
import numpy as np

def contar_aciertos(modelo, dataset):
    aciertos = 0
    fallos = 0
    for imagenes, etiquetas in dataset:
        predicciones = modelo.predict(imagenes, verbose=0).flatten()
        predicciones_binarias = (predicciones >= 0.5).astype(int)
        # TODO: suma a 'aciertos' los casos donde predicciones_binarias coincide con etiquetas
        # TODO: suma a 'fallos' los casos donde no coincide
    return aciertos, fallos

aciertos, fallos = contar_aciertos(modelo, dataset_validacion)
print(f"Aciertos: {aciertos}, Fallos: {fallos}")

<details>
<summary>💡 Ver solución</summary>

```python
def contar_aciertos(modelo, dataset):
    aciertos = 0
    fallos = 0
    for imagenes, etiquetas in dataset:
        predicciones = modelo.predict(imagenes, verbose=0).flatten()
        predicciones_binarias = (predicciones >= 0.5).astype(int)
        coincidencias = (predicciones_binarias == etiquetas.numpy())
        aciertos += int(np.sum(coincidencias))
        fallos += int(np.sum(~coincidencias))
    return aciertos, fallos
```
</details>